# absorption and Diffusion

This part is meant to apply the absorption of the metal oxide material, and solve this problem on a grid similar to the grids used in diffusion and advection equations. The first goal is to solve for:

\begin{equation}
\left\{\begin{aligned}
    \partial_t\rho&=-\dot m_a(\rho, \rho_s) \\
    \partial_t\rho_s&=\dot m_a(\rho, \rho_s)
\end{aligned}\right.
\end{equation}
Where $\rho$ is the density at that point and $\rho_s$ is the saturation of the metal. \
Under initial condition (since there are no advection/diffusion/movement we do not look at bc)
\begin{equation}
\left\{ \begin{aligned}
    \rho(x, 0) = 1 \\
    \rho_s(x,0) = 0
\end{aligned}\right.
\end{equation}

the function for calculating adsorption is given by:
$$
\dot{m}_a(\rho_s, t)
                   = C_a \exp(- \frac{E_a}{R \, T}) \, 
                        \log( \frac{p_p}{p_{eq,a}} ) \, (\rho_{sat} - \rho_s(t)) \, ,     
$$

which I will for now simplify into:

$$
\dot m(\rho, \rho_s) = k\rho(\rho_{sat}-\rho_s)
$$

if we discretize and apply forward euler we get:

$$
Mx^{n+1}=Mx^n +\dot m(x^n)\,\,\Delta t  
$$

where we need to be carefull: $x^n=(\rho^{1,n} \ldots \rho_s^{N,n})$ 

## Import Julia Packages 

In [1]:
using Ferrite, SparseArrays, WriteVTK, Plots, LinearAlgebra

In [2]:
name = "absorption_2d";

# Stop time, time step, absorption, saturation density
dt = .005;
T = 3;
k = 1.0;
rho_sat = .6;


In [3]:
# problem set-up: rhs function that defines the ODE
function drho_sdt(rho, rhos, t)
    return k * rho * (rho_sat - rhos)
end

drho_sdt (generic function with 1 method)

In [4]:
# generates 20 gridpoints between 
grid = generate_grid(Quadrilateral, (20, 20), Vec((0.0, 0.0)), Vec((1.0, 1.0)));

# generate the same 
qr = QuadratureRule{RefQuadrilateral}(2);

ip_rho = Lagrange{RefQuadrilateral, 1}();
cellvalues_rho = CellValues(qr, ip_rho);

ip_rhos = Lagrange{RefQuadrilateral, 1}()
cellvalues_rhos = CellValues(qr, ip_rhos);

# degree of freedom handler, 
dh = DofHandler(grid)
add!(dh, :rho, ip_rho)
add!(dh, :rho_s, ip_rhos)
close!(dh);

In [5]:
# for now we have no constraints
ch = ConstraintHandler(dh)

#create left_slit
# addfacetset!(
#     grid,
#     "left_slit",
#     x -> abs(x[1]) < 1e-8 && 0.25 <= x[2] <= 0.75
# )

# dbc_left = Dirichlet(
#     :rho,
#     getfacetset(grid, "left_slit"),
#     (x, t) -> 1.0
# )

# add!(ch, dbc_left)
close!(ch)
update!(ch, 0.0)

In [6]:
function doassemble_M!(M, cellvalues_rho, cellvalues_rhos, dh)
    assembler = start_assemble(M)

    # number of basis functions per field
    n_basefuncs_rho  = getnbasefunctions(cellvalues_rho)
    n_basefuncs_rhos = getnbasefunctions(cellvalues_rhos)

    # loop over all cells
    for cell in CellIterator(dh)
        # get global DOFs for this cell
        cell_dofs = celldofs(cell)
        ndofs_cell = length(cell_dofs)
        Me = zeros(ndofs_cell, ndofs_cell)

        # rho
        reinit!(cellvalues_rho, cell)
        for q in 1:getnquadpoints(cellvalues_rho)
            dx = getdetJdV(cellvalues_rho, q)
            for i in 1:n_basefuncs_rho
                psi_i = shape_value(cellvalues_rho, q, i)
                for j in 1:n_basefuncs_rho
                    psi_j = shape_value(cellvalues_rho, q, j)
                    # :rho DOFs are first in cell_dofs
                    Me[i, j] += psi_i * psi_j * dx
                end
            end
        end

        # rhos
        reinit!(cellvalues_rhos, cell)
        for q in 1:getnquadpoints(cellvalues_rhos)
            dx = getdetJdV(cellvalues_rhos, q)
            for i in 1:n_basefuncs_rhos
                psi_i = shape_value(cellvalues_rhos, q, i)
                for j in 1:n_basefuncs_rhos
                    psi_j = shape_value(cellvalues_rhos, q, j)
                    # :rho_s DOFs are after :rho in cell_dofs
                    offset = n_basefuncs_rho
                    Me[offset + i, offset + j] += psi_i * psi_j * dx
                end
            end
        end

        assemble!(assembler, cell_dofs, Me)
    end

    return M
end

doassemble_M! (generic function with 1 method)

In [7]:
for cell in CellIterator(dh)
    println(celldofs(cell))
    break
end
println(ndofs(dh))

[1, 2, 3, 4, 5, 6, 7, 8]
882


In [8]:
M = allocate_matrix(dh);
M = doassemble_M!(M, cellvalues_rho, cellvalues_rhos, dh);

A = M # + Dt .*L + dt .* K

882×882 SparseMatrixCSC{Float64, Int64} with 14884 stored entries:
⎡⠻⣦⡀⠀⢧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠈⠻⣦⠈⢧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠉⠓⠦⣄⡻⣮⡳⣄⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠙⢮⡻⣮⡳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠈⠙⢮⡻⣮⡻⣤⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠛⣮⡻⣮⡳⣄⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢮⡻⣮⡳⣦⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠹⣮⡻⣮⡳⣤⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⣮⡻⣮⡳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢮⡻⣮⡳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢮⡻⣮⡳⣆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠹⢮⡻⣮⡻⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠛⢮⡻⣮⡳⣆⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠹⢮⡻⣮⡳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠙⢮⡻⣮⡻⣤⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠛⣮⡻⣮⡳⣄⡀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢮⡻⣮⡳⣄⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠙⢮⡻⣮⠳⣤⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⣦⡻⣮⡳⣄⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢮⠻⣦⎦

In [9]:
ndofs_total = ndofs(dh)
x_n = zeros(ndofs_total)


# block ordering: (rho_1, rho_s_1, rho_2, rho_s_2, ...)?? idk
# dh is your DofHandler
for cell in CellIterator(dh)
    dofs = celldofs(cell)  # global DOFs for this cell

    # number of DOFs per field in this cell
    n_rho   = getnbasefunctions(cellvalues_rho)
    n_rhos  = getnbasefunctions(cellvalues_rhos)

    # assuming first n_rho are :rho, next n_rhos are :rho_s
    x_n[dofs[1:n_rho]] .= 1.0       # gas density = 1 initially
    x_n[dofs[n_rho+1:end]] .= 0.0   # solid initially empty
end

apply!(x_n, ch)

rhsdata = get_rhs_data(ch, A);
apply!(A, ch);

In [10]:
rho_dofs  = Int[]
rhos_dofs = Int[]

for cell in CellIterator(dh)
    cdofs = celldofs(cell)

    append!(rho_dofs,  cdofs[dof_range(dh, :rho)])
    append!(rhos_dofs, cdofs[dof_range(dh, :rho_s)])
end

rho_dofs  = unique(rho_dofs);
rhos_dofs = unique(rhos_dofs);

In [11]:
rho_hist = zeros(length(dt:dt:T));
rhos_hist = zeros(length(dt:dt:T));



In [12]:
pvd = paraview_collection(joinpath(name, name))

VTKGridFile(joinpath(name, name), dh) do vtk
    write_solution(vtk, dh, x_n)
    pvd[0.0] = vtk
end

VTKGridFile for the closed file "absorption_2d\absorption_2d.vtu".

For standard formulation a fem source/sink term would be:

$$
A\mathbf{x}^{n+1} = M\mathbf{x}^n+\mathbf{b}^n
$$

if we follow the steps the weak form for $\mathbf{b}$ becomen element wise:

$$
b^{n,j} = \int \dot m (\mathbf{x}^n)\psi^j d\Omega
$$

Now a usefull assumption to cut computational time down is:

$$
\mathbf{\dot m}^n \approx \mathbf{\dot m}_h^n = \sum \dot m^{n,i} \psi^i
$$

Where

$$
\dot m^{n,i} = k \rho^{n,i}(\rho_{sat}-\rho_s^{n,i})
$$

which gives:

\begin{equation*}
\begin{aligned}
b^{n, j} &= \int \dot m (\mathbf{x}^n)\psi^j d\Omega \\
&\approx \sum \int \dot m^{n,i} \psi^i \psi^j d\Omega\\
&= \sum \int \psi^i \psi^j d\Omega  \,\, \dot m^{n,i}\\
\mathbf{b}^n&= M\mathbf{\dot m}_h^n
\end{aligned}
\end{equation*}

In [13]:

mdot = zeros(ndofs_total)
A_lu = lu(A)
# finally the timeloop
for (step, t) in enumerate(dt:dt:T)
    
    #First of all, we need to update the Dirichlet boundary condition values.
    # does not really do anything since we do not yet apply bc's
    update!(ch, t)
    mdot = zeros(ndofs_total)
    mdot .= 0.0

    for (i_rho, i_rhos) in zip(rho_dofs, rhos_dofs)
        md = k * x_n[i_rho] * (rho_sat - x_n[i_rhos])

        mdot[i_rhos] += md
        mdot[i_rho]  -= md
    end
    
    b = M * x_n  + dt * (M * mdot)
    #Then, we can apply the boundary conditions of the current time step.
    apply_rhs!(rhsdata, b, ch)

    #Finally, we can solve the time step and save the solution afterwards.
    x = A_lu \ b

    
    VTKGridFile(joinpath(name, "$(name)_1-$step"), dh) do vtk
        write_solution(vtk, dh, x)
        pvd[t] = vtk
    end
    #At the end of the time loop, we set the previous solution to the current one and go to the next time step.
    x_n .= x
    # keep this for plotting
    rho_hist[step] = x_n[1]
    rhos_hist[step] = x_n[5]

end

In [14]:
vtk_save(pvd);

In [27]:
times = collect(dt:dt:T)

p = plot(times, rho_hist,
     xlabel = "time",
     ylabel = "density [ρ/ρ0]",
     label = "ρ",
     lw = 2)

plot!(times, rhos_hist,
      label = "ρ_s",
      lw = 2)


hline!([rho_sat],
       label = "ρ_sat",
       lw = 2,
       linestyle = :dash)

savefig(p, "..\\report\\figures\\rho_plot.png")

"c:\\Users\\noudh\\uni\\MEP\\report\\figures\\rho_plot.png"